# LeetCode 2942 — Find Words Containing Character bằng Python

Link đề: https://leetcode.com/problems/find-words-containing-character/ · Độ khó: **Easy** · Chủ đề: Array, String

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def findWordsContaining(self, words: List[str], x: str) -> List[int]:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho danh sách từ `words` và một ký tự `x`, trả về **chỉ số (index)** của những từ **có chứa** `x`, không phải chính các từ đó.
2. Biết cách **hỏi "ký tự này có nằm trong chuỗi không"** bằng toán tử `in`, và hiểu `in` chạy thế nào bên trong.
3. Biết dùng `enumerate` để lấy **chỉ số và giá trị** cùng lúc, và gom kết quả bằng `append` hoặc **list comprehension**.
4. Đi từ cách **tự duyệt từng ký tự bằng hai vòng lặp** → **dùng `in`** → **`str.find`/`str.count`** → **list comprehension + `enumerate`**, và hiểu cái nào nên chọn khi nộp bài.
5. Tránh các lỗi kinh điển: trả về **từ** thay vì **chỉ số**, dùng `words.index(w)` (sai khi có từ trùng nhau), quên `break` làm thêm chỉ số trùng, và nhầm `find` (trả `-1`) với kiểu `bool`.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `words` (đánh chỉ số từ `0`) và một ký tự `x`. Hãy trả về **mảng các chỉ số** của những từ trong `words` **chứa ký tự `x`**. Các chỉ số có thể trả về **theo thứ tự bất kỳ** (khi nộp LeetCode), nhưng ở notebook này ta trả về theo thứ tự tăng dần cho dễ so sánh.

**Ví dụ 1**

- Input: `words = ["leet", "code"]`, `x = "e"`
- Output: `[0, 1]`
- Giải thích: `"e"` xuất hiện trong `"leet"` (chỉ số 0) và `"code"` (chỉ số 1).

**Ví dụ 2**

- Input: `words = ["abc", "bcd", "aaaa", "cbc"]`, `x = "a"`
- Output: `[0, 2]`
- Giải thích: `"a"` xuất hiện trong `"abc"` (chỉ số 0) và `"aaaa"` (chỉ số 2).

**Ví dụ 3**

- Input: `words = ["abc", "bcd", "aaaa", "cbc"]`, `x = "z"`
- Output: `[]`
- Giải thích: không từ nào chứa `"z"`, nên trả về mảng rỗng.

**Ràng buộc:** `1 <= words.length <= 50`; `1 <= words[i].length <= 50`; `x` là một chữ cái tiếng Anh thường; `words[i]` chỉ gồm chữ cái tiếng Anh thường.

In [1]:
from typing import List
import random

examples = [
    (["leet", "code"], "e", [0, 1]),
    (["abc", "bcd", "aaaa", "cbc"], "a", [0, 2]),
    (["abc", "bcd", "aaaa", "cbc"], "z", []),
]
for words, x, expected in examples:
    print("words =", words, "x =", repr(x), "-> kỳ vọng", expected)

words = ['leet', 'code'] x = 'e' -> kỳ vọng [0, 1]
words = ['abc', 'bcd', 'aaaa', 'cbc'] x = 'a' -> kỳ vọng [0, 2]
words = ['abc', 'bcd', 'aaaa', 'cbc'] x = 'z' -> kỳ vọng []


## 3. Kiến thức nền

### 3.1. Chỉ số (index) của một danh sách

Danh sách (`list`) đánh số các phần tử bắt đầu từ **`0`**. Phần tử đầu có chỉ số `0`, phần tử thứ hai có chỉ số `1`, ... Đề yêu cầu trả về **các con số này**, không phải nội dung của từ.

In [2]:
words = ["leet", "code", "abc"]
print(words[0], words[1], words[2])     # leet code abc
print(len(words))                        # 3 phần tử -> chỉ số hợp lệ là 0, 1, 2
print(words[-1])                         # chỉ số âm đếm từ cuối: abc

leet code abc
3
abc


### 3.2. Chuỗi cũng là một dãy ký tự

Một chuỗi (`str`) giống một dãy các ký tự: có `len`, có thể truy cập `s[i]`, và **duyệt được bằng `for`**. Mỗi ký tự lại là một chuỗi dài `1`.

In [3]:
s = "leet"
print(len(s), s[0], s[-1])               # 4 l t
for ch in s:
    print(ch, end=" ")                   # l e e t
print()
print(type(s[0]), s[0] == "l")           # <class 'str'> True

4 l t
l e e t 
<class 'str'> True


### 3.3. Toán tử `in`: kiểm tra "có chứa"

`x in s` trả về `True` nếu ký tự (hoặc chuỗi con) `x` **xuất hiện ở đâu đó** trong `s`, ngược lại `False`. Nó phân biệt **chữ hoa/chữ thường**, và chuỗi rỗng `""` luôn là con của mọi chuỗi.

In [4]:
print("e" in "leet")                     # True
print("z" in "leet")                     # False
print("E" in "leet")                     # False: phân biệt hoa/thường
print("ee" in "leet")                    # True: cũng dùng được với chuỗi con
print("" in "leet")                      # True

True
False
False
True
True


### 3.4. `enumerate`: lấy cả chỉ số lẫn giá trị

Khi vừa cần **chỉ số `i`** vừa cần **phần tử `w`**, thay vì `for i in range(len(words))` rồi `words[i]`, ta viết `for i, w in enumerate(words)`. Ngắn hơn và ít lỗi hơn.

In [5]:
words = ["leet", "code", "abc"]
for i, w in enumerate(words):
    print(i, w)
# 0 leet
# 1 code
# 2 abc
print(list(enumerate(words)))            # [(0, 'leet'), (1, 'code'), (2, 'abc')]

0 leet
1 code
2 abc
[(0, 'leet'), (1, 'code'), (2, 'abc')]


### 3.5. Gom kết quả: `append` và list comprehension

Hai cách tạo danh sách kết quả:

- Vòng `for` + `ans.append(i)` khi điều kiện đúng;
- **List comprehension** `[i for i, w in enumerate(words) if x in w]`: đọc là "lấy `i` cho mọi cặp `(i, w)` mà `x in w`".

Hai cách cho cùng kết quả; cách sau gọn hơn.

In [6]:
words, x = ["leet", "code", "abc"], "e"

ans = []
for i, w in enumerate(words):
    if x in w:
        ans.append(i)
print(ans)                                          # [0, 1]

print([i for i, w in enumerate(words) if x in w])   # [0, 1]

[0, 1]
[0, 1]


### 3.6. `str.find` và `str.count`

- `s.find(x)` trả về **vị trí đầu tiên** của `x` trong `s`, hoặc **`-1`** nếu không có. Chú ý: vị trí `0` cũng là "có"!
- `s.count(x)` trả về **số lần** `x` xuất hiện (`0` nếu không có).

Vì `0` là một giá trị hợp lệ của `find`, **không** được viết `if s.find(x):` để kiểm tra "có".

In [7]:
print("leet".find("e"), "leet".find("z"), "leet".find("l"))   # 1 -1 0
print("leet".count("e"), "leet".count("z"))                  # 2 0
print(bool("leet".find("l")))     # False!  vì find trả 0 -> bẫy
print("leet".find("l") != -1)     # True: cách kiểm tra đúng

1 -1 0
2 0
False
True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Bài này giống việc **soát từng từ một**, hỏi "từ này có chữ `x` không?", và **ghi lại số thứ tự** của từ đó nếu có. Với `words = ["abc", "bcd", "aaaa", "cbc"]`, `x = "a"`:

| chỉ số `i` | từ `w` | `"a"` có trong `w`? | ghi lại? | `ans` sau bước này |
|---|---|---|---|---|
| 0 | `abc` | có | có | `[0]` |
| 1 | `bcd` | không | không | `[0]` |
| 2 | `aaaa` | có | có | `[0, 2]` |
| 3 | `cbc` | không | không | `[0, 2]` |

Kết quả `[0, 2]`. Vì mỗi từ độc lập với từ khác, ta chỉ cần **một lần duyệt** qua `words`; việc còn lại là cách kiểm tra "từ chứa ký tự" cho thật gọn.

## 5. Cách 1 — Brute Force / baseline: tự duyệt từng ký tự bằng hai vòng lặp

**Ý tưởng.** Chưa dùng `in`. Với mỗi từ, ta duyệt **từng ký tự** của nó; gặp ký tự bằng `x` thì đánh dấu `found = True`, ghi lại chỉ số và **`break`** (không cần xét tiếp vì đã biết từ này chứa `x`).

In [8]:
def find_words_manual(words: List[str], x: str) -> List[int]:
    ans = []
    for i in range(len(words)):              # duyệt từng từ theo chỉ số
        found = False
        for ch in words[i]:                  # duyệt từng ký tự của từ
            if ch == x:
                found = True
                break                        # đã thấy, khỏi xét tiếp
        if found:
            ans.append(i)
    return ans


for words, x, expected in examples:
    print(words, repr(x), "->", find_words_manual(words, x), "(kỳ vọng", expected, ")")

['leet', 'code'] 'e' -> [0, 1] (kỳ vọng [0, 1] )
['abc', 'bcd', 'aaaa', 'cbc'] 'a' -> [0, 2] (kỳ vọng [0, 2] )
['abc', 'bcd', 'aaaa', 'cbc'] 'z' -> [] (kỳ vọng [] )


In [9]:
# Xem hai vòng lặp làm việc: in ký tự nào được so sánh với x
words, x = ["abc", "bcd", "aaaa"], "a"
for i in range(len(words)):
    compared = []
    for ch in words[i]:
        compared.append(ch)
        if ch == x:
            break
    print(f"từ {i} = {words[i]!r}: đã so sánh {compared}")

từ 0 = 'abc': đã so sánh ['a']
từ 1 = 'bcd': đã so sánh ['b', 'c', 'd']
từ 2 = 'aaaa': đã so sánh ['a']


**Độ phức tạp (Cách 1).** Gọi `n` là số từ, `L` là độ dài từ dài nhất. Trong tệ nhất ta so sánh mọi ký tự của mọi từ: **Time `O(n · L)`**. Ngoài mảng kết quả, chỉ dùng vài biến: **Space `O(1)`** phụ (không tính đầu ra).

**Nút thắt.** Code đúng, nhưng dài dòng: phải tự quản lý cờ `found` và `break`. Python đã có sẵn toán tử làm đúng việc này.

## 6. Ý tưởng thuật toán / bước đột phá

Phép "duyệt ký tự rồi so sánh, thấy thì dừng" chính là **toán tử `in`** của Python: `x in w`. Nó làm đúng như Cách 1 nhưng được viết sẵn (và chạy bằng code C bên dưới nên nhanh hơn). Khi đó mỗi từ chỉ còn một câu hỏi, bài trở thành **"lọc các chỉ số thỏa điều kiện"**:

> `ans = [i for i, w in enumerate(words) if x in w]`

Mức độ phức tạp thời gian vẫn là `O(n · L)` (phải nhìn ký tự để biết), nhưng code ngắn, khó sai.

## 7. Các cách tiếp theo

### Cách 2 — Dùng `in` trong vòng `for` + `enumerate`

**Giải thích.** Thay hai vòng lặp bằng một vòng `for i, w in enumerate(words)` và điều kiện `if x in w`. Mọi thứ còn lại giữ nguyên: đúng thì `append(i)`.

In [10]:
def find_words_in(words: List[str], x: str) -> List[int]:
    ans = []
    for i, w in enumerate(words):
        if x in w:                           # toán tử in thay cho vòng lặp ký tự
            ans.append(i)
    return ans


for words, x, expected in examples:
    print(words, repr(x), "->", find_words_in(words, x), "(kỳ vọng", expected, ")")

['leet', 'code'] 'e' -> [0, 1] (kỳ vọng [0, 1] )
['abc', 'bcd', 'aaaa', 'cbc'] 'a' -> [0, 2] (kỳ vọng [0, 2] )
['abc', 'bcd', 'aaaa', 'cbc'] 'z' -> [] (kỳ vọng [] )


**Độ phức tạp Cách 2:** **Time `O(n · L)`**, **Space `O(1)`** phụ (không tính đầu ra).

### Cách 3 — `str.find` / `str.count`

**Giải thích.** Có thể thay `x in w` bằng `w.find(x) != -1` hoặc `w.count(x) > 0`. Cách này hữu ích khi bạn **cần thêm thông tin**: ví dụ `find` cho biết **vị trí đầu tiên**, `count` cho biết **số lần xuất hiện**. Nhưng nếu chỉ cần "có/không", `in` rõ ràng nhất. Nhớ: `find` trả `-1` khi không thấy, và `0` vẫn là "có".

In [11]:
def find_words_find(words: List[str], x: str) -> List[int]:
    ans = []
    for i, w in enumerate(words):
        if w.find(x) != -1:                  # != -1, KHÔNG phải truthy/falsy
            ans.append(i)
    return ans


def find_words_count(words: List[str], x: str) -> List[int]:
    return [i for i, w in enumerate(words) if w.count(x) > 0]


for words, x, expected in examples:
    print(find_words_find(words, x), find_words_count(words, x), "(kỳ vọng", expected, ")")

[0, 1] [0, 1] (kỳ vọng [0, 1] )
[0, 2] [0, 2] (kỳ vọng [0, 2] )
[] [] (kỳ vọng [] )


In [12]:
# Thêm thông tin nhờ find/count: vị trí đầu tiên và số lần xuất hiện
words, x = ["abc", "bcd", "aaaa", "cbc"], "a"
for i, w in enumerate(words):
    print(f"{w!r}: find={w.find(x):>2} count={w.count(x)}")

'abc': find= 0 count=1
'bcd': find=-1 count=0
'aaaa': find= 0 count=4
'cbc': find=-1 count=0


**Độ phức tạp Cách 3:** **Time `O(n · L)`**, **Space `O(1)`** phụ. `count` luôn đi hết từ (không dừng sớm), còn `in`/`find` dừng ngay khi thấy: với bài này `in` nhỉnh hơn một chút.

### Cách 4 — List comprehension một dòng (và `filter`)

**Giải thích.** Gộp vòng `for`, điều kiện và `append` thành một biểu thức: `[i for i, w in enumerate(words) if x in w]`. Một biến thể dùng `filter(lambda i: x in words[i], range(len(words)))` cho cùng kết quả, nhưng khó đọc hơn nên ít khi cần.

In [13]:
def find_words_comp(words: List[str], x: str) -> List[int]:
    return [i for i, w in enumerate(words) if x in w]


def find_words_filter(words: List[str], x: str) -> List[int]:
    return list(filter(lambda i: x in words[i], range(len(words))))


for words, x, expected in examples:
    print(find_words_comp(words, x), find_words_filter(words, x), "(kỳ vọng", expected, ")")

[0, 1] [0, 1] (kỳ vọng [0, 1] )
[0, 2] [0, 2] (kỳ vọng [0, 2] )
[] [] (kỳ vọng [] )


In [14]:
# Code nộp LeetCode cho Cách 4 (ngắn, rõ, đủ nhanh)
from typing import List

class Solution:
    def findWordsContaining(self, words: List[str], x: str) -> List[int]:
        return [i for i, w in enumerate(words) if x in w]

**Độ phức tạp Cách 4:** **Time `O(n · L)`**, **Space `O(k)`** cho `k` chỉ số trả về (không tính là bộ nhớ phụ).

## 8. TRACE từng bước

Ta theo dõi Cách 2 với `words = ["abc", "bcd", "aaaa", "cbc"]`, `x = "a"` và Cách 1 trên từng ký tự.

In [15]:
def trace_in(words, x):
    ans = []
    print("words =", words, "| x =", repr(x))
    for i, w in enumerate(words):
        has = x in w
        if has:
            ans.append(i)
        print(f"i={i} w={w!r:7} {x!r} in w -> {has!s:5} | ans={ans}")
    print("=> kết quả:", ans)
    return ans


assert trace_in(["abc", "bcd", "aaaa", "cbc"], "a") == [0, 2]

words = ['abc', 'bcd', 'aaaa', 'cbc'] | x = 'a'
i=0 w='abc'   'a' in w -> True  | ans=[0]
i=1 w='bcd'   'a' in w -> False | ans=[0]
i=2 w='aaaa'  'a' in w -> True  | ans=[0, 2]
i=3 w='cbc'   'a' in w -> False | ans=[0, 2]
=> kết quả: [0, 2]


In [16]:
def trace_manual(words, x):
    ans = []
    for i in range(len(words)):
        found = False
        steps = []
        for ch in words[i]:
            steps.append(ch)
            if ch == x:
                found = True
                break
        if found:
            ans.append(i)
        print(f"i={i} w={words[i]!r:7} đã đọc {steps!s:28} found={found!s:5} ans={ans}")
    return ans


assert trace_manual(["leet", "code"], "e") == [0, 1]

i=0 w='leet'  đã đọc ['l', 'e']                   found=True  ans=[0]
i=1 w='code'  đã đọc ['c', 'o', 'd', 'e']         found=True  ans=[0, 1]


Bảng tóm tắt (Cách 2, `x = "a"`):

| `i` | `w` | `"a" in w` | `ans` |
|---|---|---|---|
| 0 | `abc` | True | `[0]` |
| 1 | `bcd` | False | `[0]` |
| 2 | `aaaa` | True | `[0, 2]` |
| 3 | `cbc` | False | `[0, 2]` |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 4. Đúng class `Solution`, đúng method `findWordsContaining(self, words: List[str], x: str) -> List[int]`.

In [17]:
from typing import List

class Solution:
    def findWordsContaining(self, words: List[str], x: str) -> List[int]:
        ans = []
        for i, w in enumerate(words):            # i là chỉ số, w là từ
            if x in w:                           # từ có chứa x
                ans.append(i)                    # ghi lại CHỈ SỐ, không phải từ
        return ans


sol = Solution()
for words, x, expected in examples:
    print(words, repr(x), "->", sol.findWordsContaining(words, x))

['leet', 'code'] 'e' -> [0, 1]
['abc', 'bcd', 'aaaa', 'cbc'] 'a' -> [0, 2]
['abc', 'bcd', 'aaaa', 'cbc'] 'z' -> []


## 10. Vì sao thuật toán đúng

- **Mỗi từ được xét đúng một lần.** `enumerate` đi qua mọi chỉ số `0..n-1`, mỗi chỉ số một lần, nên không bỏ sót và không trùng lặp.
- **Điều kiện khớp đề.** `x in w` đúng khi và chỉ khi có ít nhất một ký tự của `w` bằng `x`, tức đúng "từ chứa ký tự `x`".
- **Thứ tự tăng dần.** Ta `append` chỉ số theo thứ tự duyệt `0, 1, 2, ...` nên kết quả luôn sắp tăng (đề cho phép thứ tự bất kỳ, tăng dần là một lựa chọn hợp lệ).
- **Không có từ nào thỏa → mảng rỗng.** `ans` khởi tạo `[]` và chỉ thay đổi khi có từ khớp.
- **Từ trùng nhau vẫn đúng.** Vì ta ghi **chỉ số** (không tra ngược từ ra chỉ số), hai từ giống hệt nhau ở hai vị trí sẽ cho hai chỉ số khác nhau.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(words)`, `L` = độ dài từ dài nhất.

- **Cách 1 (hai vòng lặp):** Time `O(n · L)`, Space `O(1)` phụ.
- **Cách 2 (`in` + `enumerate`):** Time `O(n · L)`, Space `O(1)` phụ.
- **Cách 3 (`find`/`count`):** Time `O(n · L)`, Space `O(1)` phụ.
- **Cách 4 (list comprehension):** Time `O(n · L)`, Space `O(1)` phụ (mảng kết quả là đầu ra).
- Mọi cách đều phải nhìn tới ký tự của từ (không thể biết từ có `x` mà không đọc nó), nên **không thể tốt hơn `O(n · L)`** trong tệ nhất. Ở đây `n, L <= 50`, tối đa 2 500 phép so sánh.

## 12. Bộ test

In [18]:
sol = Solution()
impls = {
    "manual": find_words_manual,
    "in": find_words_in,
    "find": find_words_find,
    "count": find_words_count,
    "comp": find_words_comp,
    "filter": find_words_filter,
    "submit": sol.findWordsContaining,
}

for words, x, expected in examples:
    for key, fn in impls.items():
        assert sorted(fn(list(words), x)) == expected, (key, words, x, fn(list(words), x), expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [19]:
def check_all(words, x, expected, label):
    for key, fn in impls.items():
        got = sorted(fn(list(words), x))
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all(["a"], "a", [0], "một từ một chữ, khớp")
check_all(["a"], "b", [], "một từ một chữ, không khớp")
check_all(["aaaa", "aaaa"], "a", [0, 1], "hai từ giống nhau đều khớp (từ trùng)")
check_all(["ab", "ab", "cd", "ab"], "b", [0, 1, 3], "từ trùng ở nhiều vị trí: phải ra mọi chỉ số")
check_all(["x"] * 50, "x", list(range(50)), "50 từ, mọi từ đều khớp")
check_all(["abc"] * 50, "z", [], "50 từ, không từ nào khớp")
check_all(["zzzz" * 12 + "ab", "b"], "a", [0], "ký tự ở cuối từ dài")

PASS một từ một chữ, khớp -> [0]
PASS một từ một chữ, không khớp -> []
PASS hai từ giống nhau đều khớp (từ trùng) -> [0, 1]
PASS từ trùng ở nhiều vị trí: phải ra mọi chỉ số -> [0, 1, 3]
PASS 50 từ, mọi từ đều khớp -> [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]
PASS 50 từ, không từ nào khớp -> []
PASS ký tự ở cuối từ dài -> [0]


### 12.2. Tính chất và phản ví dụ

In [20]:
# Hàm kiểm tra hợp lệ: đề cho phép thứ tự bất kỳ nên kiểm tra theo định nghĩa
def is_valid_answer(words, x, ans):
    if len(set(ans)) != len(ans):
        return False                                      # không trùng chỉ số
    if any(not isinstance(i, int) or not (0 <= i < len(words)) for i in ans):
        return False                                      # chỉ số hợp lệ
    return set(ans) == {i for i in range(len(words)) if x in words[i]}


assert is_valid_answer(["leet", "code"], "e", [1, 0])     # thứ tự bất kỳ vẫn hợp lệ
assert not is_valid_answer(["leet", "code"], "e", [0])    # thiếu
assert not is_valid_answer(["leet", "code"], "e", [0, 0, 1])  # trùng
assert not is_valid_answer(["leet", "code"], "e", ["leet", "code"])  # sai kiểu: từ thay vì chỉ số
print("PASS hàm kiểm tra hợp lệ")

# Phản ví dụ 1: trả từ thay vì chỉ số
def sai_tra_tu(words, x):
    return [w for w in words if x in w]

assert sai_tra_tu(["leet", "code"], "e") == ["leet", "code"] != [0, 1]
print("PASS phản ví dụ: trả từ thay vì chỉ số ->", sai_tra_tu(["leet", "code"], "e"))

# Phản ví dụ 2: dùng words.index(w) khi có từ trùng nhau
def sai_index(words, x):
    return [words.index(w) for w in words if x in w]

assert sai_index(["ab", "ab"], "a") == [0, 0] and sol.findWordsContaining(["ab", "ab"], "a") == [0, 1]
print("PASS phản ví dụ: words.index cho", sai_index(["ab", "ab"], "a"), "thay vì [0, 1]")

# Phản ví dụ 3: kiểm tra find bằng truthy (bỏ sót vị trí 0)
def sai_find_truthy(words, x):
    return [i for i, w in enumerate(words) if w.find(x)]

assert sai_find_truthy(["abc", "bca"], "a") == [1] and sol.findWordsContaining(["abc", "bca"], "a") == [0, 1]
print("PASS phản ví dụ: if w.find(x) bỏ sót từ có x ở vị trí 0 ->", sai_find_truthy(["abc", "bca"], "a"))

PASS hàm kiểm tra hợp lệ
PASS phản ví dụ: trả từ thay vì chỉ số -> ['leet', 'code']
PASS phản ví dụ: words.index cho [0, 0] thay vì [0, 1]
PASS phản ví dụ: if w.find(x) bỏ sót từ có x ở vị trí 0 -> [1]


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [21]:
random.seed(2942)
letters = "abcdefghijklmnopqrstuvwxyz"
for trial in range(2000):
    n = random.randint(1, 50)
    alphabet = random.choice([letters[:2], letters[:5], letters])      # bảng chữ nhỏ -> nhiều trùng/khớp
    words = ["".join(random.choice(alphabet) for _ in range(random.randint(1, 50))) for _ in range(n)]
    x = random.choice(letters)
    ref = find_words_manual(list(words), x)
    for key, fn in impls.items():
        got = fn(list(words), x)
        assert got == ref, (key, words, x)
        assert is_valid_answer(words, x, got)
print("PASS 2000 bộ ngẫu nhiên - mọi cách khớp nhau và hợp lệ")

PASS 2000 bộ ngẫu nhiên - mọi cách khớp nhau và hợp lệ


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Trả về từ thay vì chỉ số

In [22]:
def sai_tu(words, x):
    return [w for w in words if x in w]          # trả về từ


print("sai  :", sai_tu(["leet", "code"], "e"), "| đúng:", Solution().findWordsContaining(["leet", "code"], "e"))

sai  : ['leet', 'code'] | đúng: [0, 1]


**Vì sao sai:** đề yêu cầu **chỉ số** (`[0, 1]`), không phải nội dung từ. Cần dùng `enumerate` để có `i`.

### Lỗi 2: Dùng `words.index(w)` để lấy chỉ số

In [23]:
def sai_index_2(words, x):
    ans = []
    for w in words:
        if x in w:
            ans.append(words.index(w))           # index tìm lần xuất hiện ĐẦU TIÊN của w
    return ans


print("sai  :", sai_index_2(["ab", "cd", "ab"], "a"), "| đúng:", Solution().findWordsContaining(["ab", "cd", "ab"], "a"))

sai  : [0, 0] | đúng: [0, 2]


**Vì sao sai:** `words.index(w)` luôn trả chỉ số của lần xuất hiện **đầu tiên** của `w`. Nếu `"ab"` xuất hiện ở chỉ số 0 và 2, ta nhận `[0, 0]` thay vì `[0, 2]`. Ngoài ra nó còn phải quét lại danh sách nên chậm. Hãy dùng `enumerate`.

### Lỗi 3: Dùng `if w.find(x):` làm điều kiện

In [24]:
def sai_find_2(words, x):
    ans = []
    for i, w in enumerate(words):
        if w.find(x):                            # find trả 0 khi x ở đầu từ -> bị coi là False
            ans.append(i)
    return ans


print("sai  :", sai_find_2(["abc", "xay"], "a"), "| đúng:", Solution().findWordsContaining(["abc", "xay"], "a"))

sai  : [1] | đúng: [0, 1]


**Vì sao sai:** `find` trả `0` nếu `x` ở đầu từ (đúng là "có") nhưng `0` là falsy trong Python, còn `-1` (không có) lại là truthy. Điều kiện bị đảo hoàn toàn. Hãy viết `w.find(x) != -1` hoặc đơn giản `x in w`.

### Lỗi 4: Quên `break` ở cách tự duyệt ký tự

In [25]:
def sai_khong_break(words, x):
    ans = []
    for i, w in enumerate(words):
        for ch in w:
            if ch == x:
                ans.append(i)                    # quên break: gặp nhiều lần là thêm nhiều lần
    return ans


print("sai  :", sai_khong_break(["aaaa", "ab"], "a"), "| đúng:", Solution().findWordsContaining(["aaaa", "ab"], "a"))

sai  : [0, 0, 0, 0, 1] | đúng: [0, 1]


**Vì sao sai:** từ `"aaaa"` có 4 ký tự `a`, mỗi lần gặp lại `append(0)` nên chỉ số `0` bị lặp 4 lần. Phải `break` sau khi ghi (hoặc dùng `in` để tránh hẳn lỗi này).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "trả về **chỉ số** của các phần tử thỏa điều kiện" → `[i for i, v in enumerate(a) if cond(v)]`.
- "Chuỗi **có chứa** ký tự/chuỗi con" → `x in s`. Chỉ dùng `find` khi cần vị trí, `count` khi cần số lần.
- Phân biệt: `in` với **list** thì so sánh **cả phần tử** (`"e" in ["leet"]` là `False`), còn `in` với **str** là kiểm tra **chuỗi con**.
- Mảng kết quả có thể trả về theo thứ tự bất kỳ thì **vẫn nên trả tăng dần**: dễ test, dễ debug.
- Nếu phải hỏi "có chứa" **nhiều lần** với nhiều ký tự khác nhau cho cùng danh sách từ, hãy tiền xử lý mỗi từ thành `set(w)` để mỗi truy vấn là `O(1)` cho mỗi từ.

## 15. Biến thể của bài

1. Trả về **các từ** (không phải chỉ số) chứa `x`.
2. Tìm các từ chứa **tất cả** ký tự trong một chuỗi `chars`: `set(chars) <= set(w)`.
3. Tìm các từ chứa `x` **ít nhất `k` lần**: `w.count(x) >= k`.
4. Với **nhiều** ký tự truy vấn, trả về dict `ký tự -> danh sách chỉ số` (chỉ mục ngược, *inverted index*).
5. Trả về chỉ số của từ **không** chứa `x`.

Thử biến thể 3 và 4:

In [26]:
def words_with_at_least_k(words, x, k):
    return [i for i, w in enumerate(words) if w.count(x) >= k]


def inverted_index(words):
    index = {}
    for i, w in enumerate(words):
        for ch in set(w):                        # set để mỗi từ chỉ ghi một lần cho mỗi ký tự
            index.setdefault(ch, []).append(i)
    return index


words = ["abc", "bcd", "aaaa", "cbc"]
print("3. 'a' xuất hiện >= 2 lần:", words_with_at_least_k(words, "a", 2))     # [2]
idx = inverted_index(words)
print("4.", {ch: idx[ch] for ch in sorted(idx)})
for x in "abcdz":
    assert idx.get(x, []) == Solution().findWordsContaining(words, x)
print("PASS chỉ mục ngược khớp lời giải gốc")

3. 'a' xuất hiện >= 2 lần: [2]
4. {'a': [0, 2], 'b': [0, 1, 3], 'c': [0, 1, 3], 'd': [1]}
PASS chỉ mục ngược khớp lời giải gốc


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai vòng lặp, `found` + `break` | tự so sánh từng ký tự | `O(n·L)` | `O(1)` | học cơ chế bên trong của `in` |
| 2. `in` + `enumerate` | `x in w` | `O(n·L)` | `O(1)` | **cách nộp bài**, rõ ràng nhất |
| 3. `find` / `count` | `w.find(x) != -1`, `w.count(x) > 0` | `O(n·L)` | `O(1)` | khi còn cần vị trí / số lần |
| 4. List comprehension / `filter` | một dòng | `O(n·L)` | `O(1)` | code gọn khi đã quen cú pháp |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"e" in "code"`, `"E" in "code"`, `"code".find("d")`, `"code".find("z")`, `list(enumerate(["a", "b"]))`.

**Bài 2 (tính tay).** Với `words = ["hello", "world", "leetcode"]`, `x = "o"`: ghi bảng từng bước như mục 4 và cho biết kết quả. Làm lại với `x = "l"`.

**Bài 3 (điền TODO).** Hoàn thành `find_words_todo(words, x)`.

In [27]:
def find_words_todo(words, x):
    ans = []
    for i, w in enumerate(words):
        # TODO 1: nếu x có mặt trong w
        #     TODO 2: thêm chỉ số i vào ans
        pass
    return ans

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `find_words_manual2(words, x)` **không dùng `in`, `find`, `count`**, chỉ dùng vòng lặp và so sánh `==` (như Cách 1). Kiểm tra với ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `words_without(words, x)`: chỉ số các từ **không** chứa `x`. (b) Viết `first_position(words, x)`: với mỗi từ chứa `x`, trả về cặp `(chỉ số từ, vị trí đầu tiên của x trong từ)`. (c) Viết `most_frequent_word_index(words, x)`: chỉ số từ có **nhiều chữ `x` nhất** (hòa thì lấy chỉ số nhỏ hơn); trả `-1` nếu không từ nào chứa `x`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `True`, `False`, `2`, `-1`, `[(0, 'a'), (1, 'b')]`. Bài 3: hai TODO nằm ngay trong comment: `if x in w:` rồi `ans.append(i)`.
- **Mức 2 (vừa):** Bài 2: `x = "o"`: `hello` có o (0), `world` có o (1), `leetcode` có o (2) nên `[0, 1, 2]`. `x = "l"`: cả ba đều có `l` (`hello`, `world`, `leetcode`) nên `[0, 1, 2]`. Bài 4: copy Cách 1, nhớ `break`.
- **Mức 3 (sâu):** Bài 5(a): đổi điều kiện thành `x not in w`. 5(b): dùng `w.find(x)`, chỉ ghi khi `!= -1`. 5(c): tính `w.count(x)` cho từng từ, giữ `best` và `best_i`; chỉ cập nhật khi **lớn hơn hẳn** (`>`) để hòa thì giữ chỉ số nhỏ hơn; nếu `best == 0` trả `-1`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert ("e" in "code") is True and ("E" in "code") is False
assert "code".find("d") == 2 and "code".find("z") == -1
assert list(enumerate(["a", "b"])) == [(0, "a"), (1, "b")]

# Bài 2
assert Solution().findWordsContaining(["hello", "world", "leetcode"], "o") == [0, 1, 2]
assert Solution().findWordsContaining(["hello", "world", "leetcode"], "l") == [0, 1, 2]

# Bài 3
def find_words_todo_ans(words, x):
    ans = []
    for i, w in enumerate(words):
        if x in w:
            ans.append(i)
    return ans

for words, x, expected in examples:
    assert find_words_todo_ans(words, x) == expected

# Bài 4
def find_words_manual2(words, x):
    ans = []
    for i in range(len(words)):
        for ch in words[i]:
            if ch == x:
                ans.append(i)
                break
    return ans

for words, x, expected in examples:
    assert find_words_manual2(words, x) == expected

# Bài 5(a)
def words_without(words, x):
    return [i for i, w in enumerate(words) if x not in w]

assert words_without(["abc", "bcd", "aaaa", "cbc"], "a") == [1, 3]

# Bài 5(b)
def first_position(words, x):
    return [(i, w.find(x)) for i, w in enumerate(words) if w.find(x) != -1]

assert first_position(["abc", "bcd", "aaaa", "cbc"], "c") == [(0, 2), (1, 1), (3, 0)]

# Bài 5(c)
def most_frequent_word_index(words, x):
    best, best_i = 0, -1
    for i, w in enumerate(words):
        c = w.count(x)
        if c > best:                              # lớn hơn hẳn mới đổi -> hòa giữ chỉ số nhỏ
            best, best_i = c, i
    return best_i

assert most_frequent_word_index(["abc", "bcd", "aaaa", "cbc"], "a") == 2
assert most_frequent_word_index(["abca", "aabb", "a"], "a") == 0      # hòa 2-2 -> chỉ số nhỏ
assert most_frequent_word_index(["abca", "aaab", "a"], "a") == 1
assert most_frequent_word_index(["xyz"], "a") == -1
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (lọc chỉ số theo điều kiện)

```python
from typing import List

class Solution:
    def findWordsContaining(self, words: List[str], x: str) -> List[int]:
        return [i for i, w in enumerate(words) if x in w]
```

**Checklist**

- [ ] Trả về **chỉ số** `i`, không phải từ `w`?
- [ ] Dùng `enumerate` (hoặc `range(len(...))`), không dùng `words.index(w)`?
- [ ] Kiểm tra "có chứa" bằng `x in w` (hoặc `find(x) != -1`), không dùng `if w.find(x):`?
- [ ] Nếu tự duyệt ký tự: đã `break` sau khi ghi để không trùng chỉ số?
- [ ] Đã thử: không từ nào khớp (`[]`), mọi từ khớp, từ trùng nhau, `x` ở đầu/cuối từ?
- [ ] Skeleton đúng `class Solution` / `def findWordsContaining(self, words: List[str], x: str) -> List[int]`?